# Lecture 3. Hypothesis tests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture03/lecture03.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Newbold, Carlson and Thorne, sections 9.1 to 9.4 and 9.6.

This notebook follows the lecture slides. Every test is worked through in the same steps: the question, what we know
and want to find out (including the two hypotheses), the formula and the decision rule, the calculation by hand (on
the slides), the same calculation in Python, and the answer for a manager. Parts 1 to 4 each end with an exercise.
In an exercise you predict a result, run the cell, change one value, and answer a question about what you see. You do
not need to write any code. The last section holds a summary table, the symbols and the Danish terms, so you can use
the notebook as a reference afterwards.

Run the setup cell first. It loads the RetailCorp data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # the study of 200 orders
population = pd.read_csv(url + "orders_population.csv")  # all 50,000 orders, used only in the demonstration

print(f"Sample: {len(sample)} orders. Population: {len(population):,} orders.")
sample.head()

## The case and the question

In Lecture 2 we asked which values of a parameter are plausible, given the one sample RetailCorp has. Today we ask
whether one particular value is plausible: the value RetailCorp claims. RetailCorp makes three claims about its
deliveries.

**Is the gap between what the sample shows and what RetailCorp claims too large to be chance?**

| RetailCorp claim | Parameter | Part |
|---|---|---|
| Deliveries take 36 hours on average | μ | 1 and 2 |
| At most 5% of orders arrive later than 48 hours (the promise) | P | 3 |
| The standard deviation of delivery times is at most 7.3 hours | σ | 4 |

In [ ]:
n = len(sample)
x_bar = sample["delivery_hours"].mean()
s = sample["delivery_hours"].std(ddof=1)
p_hat = sample["late"].mean()

print(f"n = {n}")
print(f"x_bar = {x_bar:.3f} hours")
print(f"s = {s:.3f} hours")
print(f"p_hat = {p_hat:.3f} ({sample['late'].sum()} late orders)")

## Part 1. A mean, one-sided

**Question.** RetailCorp tells customers that delivery takes 36 hours on average. Do deliveries take longer?

**What we know.** n = 200 orders, x̄ = 36.789 hours and s = 8.140 hours. σ is unknown.

**What we want.** Whether the sample gives strong evidence that μ, the average delivery time of all orders, is above
36 hours. x̄ is 0.79 hours above 36, but a sample average always differs a little from μ (Lecture 1).

**Hypotheses.** The null hypothesis H<sub>0</sub> is the claim or the status quo, which we keep unless the sample gives strong evidence
against it. The alternative H<sub>1</sub> is what we conclude if we reject H<sub>0</sub>. Only "longer" matters here,
so the test is one-sided:

H<sub>0</sub>: μ ≤ 36 &nbsp;&nbsp;&nbsp; H<sub>1</sub>: μ > 36 &nbsp;&nbsp;&nbsp; significance level α = 0.05,
chosen before the test.

**Test statistic.** t = (x̄ − μ<sub>0</sub>) / (s/√n), where μ<sub>0</sub> = 36 (read mu-nought). If μ = 36 and the
population is normal or n is large, t follows a t distribution with n − 1 degrees of freedom.

**Decision rule.** Reject H<sub>0</sub> if t > t<sub>n−1, α</sub>, the value with α above it, or equivalently if
the p-value is below α. The p-value is the probability of a t at least as large as ours, if μ = 36. If μ = 36, t
lands above the critical value in 5% of samples; if μ is below 36, even less often.

`stats.t.ppf` takes the area to the *left*, so t<sub>n−1, α</sub> is `stats.t.ppf(1 - alpha, df=n - 1)`.
`stats.t.sf` gives the area to the *right* of a value, which is the one-sided p-value here.

In [ ]:
mu0 = 36
alpha = 0.05
se = s / np.sqrt(n)
t_stat = (x_bar - mu0) / se
t_crit = stats.t.ppf(1 - alpha, df=n - 1)
p = stats.t.sf(t_stat, df=n - 1)     # area to the right of t_stat

print(f"Standard error: {se:.4f}")
print(f"t = {t_stat:.3f}, crit = {t_crit:.3f}, p = {p:.3f}")

**Decision.** t = 1.371 is below the critical value 1.653, and p = 0.086 is above α = 0.05, so we do not reject
H<sub>0</sub>.

**For the manager.** The sample average, 36.8 hours, is above 36, but a gap of this size could well be chance
(p = 0.086). The sample does not give strong evidence that deliveries take longer than 36 hours on average. It does
not show that the claim is right either.

A check: scipy has a function that does the whole test from the data. It gives the same t and p.

In [ ]:
stats.ttest_1samp(sample["delivery_hours"], popmean=36, alternative="greater")

**What the population shows.** Because the population is simulated, we can look up the true average: μ = 36.40
hours. So H<sub>0</sub> is false, and our decision not to reject it is a Type II error. With real data we never learn
this. "Do not reject H<sub>0</sub>" does not show that H<sub>0</sub> is true; it says that the sample is not strong
evidence against it.

**If σ were known** (Newbold 9.2), the test would use σ and the standard normal with the same steps:
z = (x̄ − μ<sub>0</sub>) / (σ/√n). σ = 7.819 is known here only because the population is simulated.

In [ ]:
sigma = 7.819
z = (x_bar - mu0) / (sigma / np.sqrt(n))
print(f"z = {z:.3f}, crit = {stats.norm.ppf(1 - alpha):.3f}, p = {stats.norm.sf(z):.3f}")

### Exercise 1. Copenhagen orders only

**Predict.** Copenhagen orders average 38.2 hours, but there are only 80 of them. Will the test reject
H<sub>0</sub>: μ ≤ 36? Write down your guess.

**Run** the cell. It prints t, the critical value and p.

In [ ]:
warehouse = "Copenhagen"      # change to "Aarhus" in step 3

x = sample.loc[sample["warehouse"] == warehouse, "delivery_hours"]
n_w = len(x)
t_w = (x.mean() - mu0) / (x.std(ddof=1) / np.sqrt(n_w))
crit_w = stats.t.ppf(1 - alpha, df=n_w - 1)
p_w = stats.t.sf(t_w, df=n_w - 1)

print(f"{warehouse}: n = {n_w}, x_bar = {x.mean():.2f}, s = {x.std(ddof=1):.2f}")
print(f"t = {t_w:.3f}, crit = {crit_w:.3f}, p = {p_w:.3f}")

**Change** `warehouse = "Copenhagen"` to `"Aarhus"` and run the cell again.

**Answer.** Why can the test reject for Copenhagen when it could not for all 200 orders?

## Part 2. A mean, two-sided

**Question.** The staffing plan for the warehouses assumes 36 hours on average. Is the average different, in either
direction? Too slow and too fast both matter for the plan, so the test is two-sided:

H<sub>0</sub>: μ = 36 &nbsp;&nbsp;&nbsp; H<sub>1</sub>: μ ≠ 36

**Decision rule.** α is split, 0.025 in each tail. Reject H<sub>0</sub> if t < −t<sub>n−1, α/2</sub> or
t > t<sub>n−1, α/2</sub>. The p-value counts both tails: twice the area beyond |t|.

In [ ]:
t_crit2 = stats.t.ppf(1 - alpha / 2, df=n - 1)
p2 = 2 * stats.t.sf(abs(t_stat), df=n - 1)     # both tails
print(f"t = {t_stat:.3f}, crit = ±{t_crit2:.3f}, p = {p2:.3f}")

Same t as in Part 1, a different question. We do not reject H<sub>0</sub>.

**The test and the interval agree.** The 95% interval from Lecture 2 is 35.65 to 37.92 hours. It contains 36, and
the two-sided test at α = 0.05 does not reject μ = 36. For a mean this always holds: a two-sided test at α = 0.05
rejects μ<sub>0</sub> exactly when μ<sub>0</sub> lies outside the 95% interval.

In [ ]:
me = t_crit2 * se
print(f"95% interval for mu: [{x_bar - me:.2f}, {x_bar + me:.2f}] hours")

**What a p-value is and is not.** p = 0.172 is the probability of a t at least as far from 0 as 1.371, if the
average were 36 hours. It is not the probability that H<sub>0</sub> is true, it does not measure the size or the
importance of the difference, and p > α does not prove H<sub>0</sub>.

### Exercise 2. Last year's figure: 35 hours

**Predict.** Look at the Lecture 2 interval, 35.65 to 37.92 hours. Will a two-sided test at 5% reject
μ<sub>0</sub> = 35? Write down your guess.

**Run** the cell.

In [ ]:
mu0_e2 = 35      # change to 37 in step 3

t_e2 = (x_bar - mu0_e2) / se
p_e2 = 2 * stats.t.sf(abs(t_e2), df=n - 1)
print(f"mu0 = {mu0_e2}: t = {t_e2:.3f}, p (two-sided) = {p_e2:.4f}")

**Change** `mu0_e2 = 35` to `37` and run the cell again.

**Answer.** How could you have answered both questions from the interval alone?

### Demonstration: how often does the test go wrong?

This cell is run in the lecture. It draws 1,000 random samples of 200 orders from the population and runs the
one-sided t test at α = 0.05 on each, twice: once testing the true mean, where H<sub>0</sub> is true, and once testing
36 hours, where H<sub>0</sub> is false because the true mean is 36.40. We can do this only because the population is
simulated.

In [ ]:
n_demo = 200
reps = 1000
rng = np.random.default_rng(2)
pop = population["delivery_hours"].to_numpy()
mu = pop.mean()
t_crit_demo = stats.t.ppf(0.95, df=n_demo - 1)

reject_true = 0      # testing mu0 = the true mean (H0 true)
reject_36 = 0        # testing mu0 = 36 (H0 false)
for i in range(reps):
    x = rng.choice(pop, size=n_demo, replace=False)
    se_i = x.std(ddof=1) / np.sqrt(n_demo)
    if (x.mean() - mu) / se_i > t_crit_demo:
        reject_true += 1
    if (x.mean() - 36) / se_i > t_crit_demo:
        reject_36 += 1

print(f"True mean: {mu:.2f} hours")
print(f"Testing mu0 = {mu:.2f} (H0 true): H0 rejected in {reject_true / reps:.1%} of samples")
print(f"Testing mu0 = 36 (H0 false): H0 rejected in {reject_36 / reps:.1%} of samples")

When H<sub>0</sub> is true, the test rejects in about 5% of samples: the Type I error rate is α. When the true average
is 0.40 hours above 36, the test rejects in only about 18% of samples, so in about 82% it misses the difference: a
Type II error. A larger sample or a larger difference makes Type II errors rarer.

## Part 3. A proportion

**Question.** RetailCorp promises that at most 5% of orders arrive later than 48 hours. Does it break the promise?

**What we know.** n = 200 orders, 16 of them late, so p̂ = 0.080. In Lecture 2 the 95% interval, 0.042 to 0.118,
contained 0.05 and could not settle the question.

**Hypotheses.** H<sub>0</sub> is the promise. The promise is broken only if P is above 0.05, so the test is one-sided:

H<sub>0</sub>: P ≤ 0.05 &nbsp;&nbsp;&nbsp; H<sub>1</sub>: P > 0.05 &nbsp;&nbsp;&nbsp; α = 0.05

**Test statistic.** z = (p̂ − P<sub>0</sub>) / √(P<sub>0</sub>(1 − P<sub>0</sub>)/n), with P<sub>0</sub> = 0.05
(read P-nought). If P = P<sub>0</sub> and nP<sub>0</sub>(1 − P<sub>0</sub>) > 5, z is approximately standard
normal (Newbold 9.4). Reject H<sub>0</sub> if z > z<sub>α</sub> = 1.645.

In [ ]:
p0 = 0.05
p_hat = sample["late"].mean()
print(f"Condition n * p0 * (1 - p0): {n * p0 * (1 - p0):.1f}   (above 5)")

se_p0 = np.sqrt(p0 * (1 - p0) / n)    # standard error with P0, the value in H0
z = (p_hat - p0) / se_p0
p = stats.norm.sf(z)
print(f"Standard error: {se_p0:.5f}")
print(f"z = {z:.3f}, crit = {stats.norm.ppf(1 - alpha):.3f}, p = {p:.3f}")

**Decision.** z = 1.947 is above 1.645 and p = 0.026 is below 0.05, so we reject H<sub>0</sub>.

**For the manager.** The sample points against the promise at the 5% level: 16 of 200 orders were late, against 10
expected if exactly 5% of all orders were late (p = 0.026). The p-value is close to the 5% line, so a larger study
would give a firmer answer.

**Which standard error?** The test asks what the data would look like if H<sub>0</sub> were true, so it computes the
standard error with P<sub>0</sub> = 0.05. The interval in Lecture 2 has no P<sub>0</sub> to assume, so it used p̂.
With the larger standard error and α split over two tails, the interval could not settle the promise. The test needs
both changes to reject: with P<sub>0</sub> but two-sided, z = 1.947 is below 1.960 (p = 0.052).

In [ ]:
se_phat = np.sqrt(p_hat * (1 - p_hat) / n)
print(f"Standard error with P0:    {se_p0:.4f}")
print(f"Standard error with p_hat: {se_phat:.4f}")

There is more than one way to compute this standard error. Some Python functions use p̂ in the test as well, for
example `proportions_ztest` in statsmodels by default (see the reference section). Here that gives z = 1.56 and
p = 0.059, a different decision. In this course we follow Newbold and use P<sub>0</sub>. In your notebook and at the
exam, know which standard error you use and be able to say why.

### Exercise 3. Late orders from Aarhus

**Predict.** 6 of the 120 Aarhus orders were late. What will z and p be? Write down your guess.

**Run** the cell. It also prints the condition.

In [ ]:
warehouse = "Aarhus"      # change to "Copenhagen" in step 3

x = sample.loc[sample["warehouse"] == warehouse, "late"]
n_w = len(x)
p_w = x.mean()
print(f"{warehouse}: n = {n_w}, late = {x.sum()}, p_hat = {p_w:.3f}")
print(f"Condition n * p0 * (1 - p0): {n_w * p0 * (1 - p0):.1f}")

z_w = (p_w - p0) / np.sqrt(p0 * (1 - p0) / n_w)
print(f"z = {z_w:.3f}, p = {stats.norm.sf(z_w):.3f}")

**Change** `warehouse = "Aarhus"` to `"Copenhagen"` and run the cell again. Look at the condition. The cell prints
z and p in any case; whether they can be used depends on the condition.

**Answer.** What can RetailCorp say about Aarhus and the promise? Should it use this test for Copenhagen?

## Part 4. A variance

**Question.** Operations aims for a standard deviation of delivery times of at most 7.3 hours. Is the spread larger?
Why 7.3 hours: with an average of 36 hours and normal delivery times, a larger standard deviation puts more than 5% of
orders above 48 hours, because (48 − 36)/1.645 ≈ 7.3.

**What we know.** n = 200 orders, s² = 66.257 hours², s = 8.140 hours.

**Hypotheses.** H<sub>0</sub>: σ² ≤ 53.29 (that is, 7.3²) &nbsp;&nbsp;&nbsp; H<sub>1</sub>: σ² > 53.29
&nbsp;&nbsp;&nbsp; α = 0.05

**Test statistic.** χ² = (n − 1)s² / σ<sub>0</sub>². If σ² = 53.29 and the population is normal, χ²
follows a chi-square distribution with n − 1 degrees of freedom (Lecture 1). Reject H<sub>0</sub> if
χ² > χ²<sub>n−1, α</sub>. The test holds only when the population is normal; a large n does not help. Delivery times
passed the check in Lecture 2: the histogram below is roughly symmetric and bell-shaped.

In [ ]:
plt.hist(sample["delivery_hours"], bins=20, edgecolor="white")
plt.xlabel("Delivery time (hours)")
plt.title("Delivery times of the 200 orders")
plt.show()
print(f"Skewness: {stats.skew(sample['delivery_hours']):.2f}")

In [ ]:
sigma0 = 7.3
s2 = sample["delivery_hours"].var(ddof=1)
chi2 = (n - 1) * s2 / sigma0**2
crit = stats.chi2.ppf(1 - alpha, df=n - 1)
p = stats.chi2.sf(chi2, df=n - 1)     # area to the right

print(f"(n - 1) s^2 = {(n - 1) * s2:.1f}, sigma0^2 = {sigma0**2:.2f}")
print(f"chi2 = {chi2:.2f}, crit = {crit:.2f}, p = {p:.3f}")

**Decision.** χ² = 247.42 is above 232.91 and p = 0.011, so we reject H<sub>0</sub>.

**For the manager.** The sample gives strong evidence that delivery times vary more than the 7.3-hour target allows
(p = 0.011). If σ is above 7.3 hours, more than 5% of orders would arrive after 48 hours even at an average of 36
hours.

### Exercise 4. Copenhagen orders only

**Predict.** Copenhagen has s = 8.43 hours, but from only 80 orders. Will the test reject H<sub>0</sub>: σ ≤ 7.3?
Write down your guess.

**Run** the cell.

In [ ]:
sigma0_e4 = 7.3      # change to 8.0 in step 3

x = sample.loc[sample["warehouse"] == "Copenhagen", "delivery_hours"]
n_w = len(x)
chi2_w = (n_w - 1) * x.var(ddof=1) / sigma0_e4**2
crit_w = stats.chi2.ppf(1 - alpha, df=n_w - 1)
print(f"Copenhagen: n = {n_w}, s = {x.std(ddof=1):.2f}")
print(f"chi2 = {chi2_w:.2f}, crit = {crit_w:.2f}, p = {stats.chi2.sf(chi2_w, df=n_w - 1):.3f}")

**Change** `sigma0_e4 = 7.3` to `8.0` and run the cell again.

**Answer.** What does the result for 8.0 tell RetailCorp, and what does it not tell?

## Reference

Every calculation from the slides is in the parts above. This section adds the statsmodels check and the tables to
revise from.

### The proportion test in statsmodels

`proportions_ztest` from statsmodels computes the standard error with p̂ unless you give `prop_var`. With
`prop_var=0.05` it uses P<sub>0</sub>, as Newbold and this course do. The cell shows both for the promise test.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

late = int(sample["late"].sum())
z_default, p_default = proportions_ztest(late, n, value=0.05, alternative="larger")
z_p0, p_p0 = proportions_ztest(late, n, value=0.05, alternative="larger", prop_var=0.05)
print(f"Default (standard error with p_hat): z = {z_default:.3f}, p = {p_default:.3f}")
print(f"prop_var=0.05 (standard error with P0): z = {z_p0:.3f}, p = {p_p0:.3f}")

### Three tests

Is the gap between what the sample shows and what RetailCorp claims too large to be chance?

| Parameter | RetailCorp H<sub>0</sub> | Test statistic | At the H<sub>0</sub> value | Holds when | Result, α = 0.05 |
|---|---|---|---|---|---|
| μ (Parts 1 and 2) | μ ≤ 36 | t = (x̄ − μ<sub>0</sub>) / (s/√n) | t with n − 1 degrees of freedom | Population normal, or n large | t = 1.371, p = 0.086: do not reject |
| P (Part 3) | P ≤ 0.05 | z = (p̂ − P<sub>0</sub>) / √(P<sub>0</sub>(1 − P<sub>0</sub>)/n) | Approximately standard normal | nP<sub>0</sub>(1 − P<sub>0</sub>) > 5 | z = 1.947, p = 0.026: reject |
| σ² (Part 4) | σ² ≤ 53.29 | χ² = (n − 1)s² / σ<sub>0</sub>² | Chi-square with n − 1 degrees of freedom | Population normal. A large n does not help. | χ² = 247.42, p = 0.011: reject |

With σ known (Newbold 9.2), the test for a mean uses z = (x̄ − μ<sub>0</sub>) / (σ/√n) and the standard normal.

### Errors

| | Do not reject H<sub>0</sub> | Reject H<sub>0</sub> |
|---|---|---|
| H<sub>0</sub> true | Correct | Type I error, probability α |
| H<sub>0</sub> false | Type II error, probability β | Correct, probability 1 − β (the power) |

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| H<sub>0</sub>, H<sub>1</sub> | H-nought, H-one | Null hypothesis (the claim) and alternative hypothesis |
| μ<sub>0</sub>, P<sub>0</sub>, σ<sub>0</sub>² | mu-nought, P-nought, sigma-nought squared | The value of the parameter in H<sub>0</sub> |
| α | alpha | Significance level: the probability of rejecting a true H<sub>0</sub> |
| β | beta | Probability of not rejecting a false H<sub>0</sub> |
| z<sub>α</sub>, t<sub>n−1, α</sub>, χ²<sub>n−1, α</sub> | z alpha, t with n minus one degrees of freedom alpha, ... | Value with α above it in the distribution |
| p | p-value | Probability of a result at least as extreme as ours, if H<sub>0</sub> is true |

### Python functions

| What | Python |
|---|---|
| Critical value, one-sided (upper tail) | `stats.t.ppf(1 - alpha, df=n - 1)`, `stats.norm.ppf(1 - alpha)`, `stats.chi2.ppf(1 - alpha, df=n - 1)` |
| Critical value, H<sub>1</sub> with < | `stats.t.ppf(alpha, df=n - 1)` (negative), `stats.norm.ppf(alpha)`, `stats.chi2.ppf(alpha, df=n - 1)`, which is χ²<sub>n−1, 1−α</sub> in Newbold's notation |
| Critical value, two-sided | `stats.t.ppf(1 - alpha / 2, df=n - 1)` |
| p-value, H<sub>1</sub> with > | `stats.t.sf(t, df=n - 1)`, `stats.norm.sf(z)`, `stats.chi2.sf(chi2, df=n - 1)` |
| p-value, H<sub>1</sub> with < | `stats.t.cdf(t, df=n - 1)`, `stats.norm.cdf(z)`, `stats.chi2.cdf(chi2, df=n - 1)` |
| p-value, two-sided t or z | `2 * stats.t.sf(abs(t), df=n - 1)`, `2 * stats.norm.sf(abs(z))` |
| Check of the t test | `stats.ttest_1samp(x, popmean=mu0, alternative="greater")` (or `"less"`, `"two-sided"`) |

`ppf` gives the value with a given probability *below* it, which is why the upper-tail values in Newbold's notation
use `1 - alpha`. `sf` gives the area *above* a value and `cdf` the area below it.

### Danish terms

| English | Danish |
|---|---|
| hypothesis test | hypotesetest |
| null hypothesis, alternative hypothesis | nulhypotese, alternativ hypotese |
| significance level | signifikansniveau |
| test statistic | teststørrelse |
| critical value, rejection region | kritisk værdi, kritisk område |
| p-value | p-værdi |
| Type I error, Type II error | type I-fejl, type II-fejl |
| one-sided test, two-sided test | ensidet test, tosidet test |